# 05 — LangChain middleware, with a real Ollama classifier

The router **is** agent middleware (`wrap_model_call`), not LCEL `RunnableBranch`.
The classifier inside the hook is now a real local LLM, not regex.

- **Classifier**: `ChatOllama(model=OLLAMA_MODEL_2)` — `qwen3.5:9b` — `temperature=0`,
  wrapped in `.with_structured_output(RouteTier, method="json_schema")`. Returns a
  typed `tier` (`SIMPLE`/`MEDIUM`/`COMPLEX`/`REASONING`) plus a one-sentence `reason`,
  not free text to parse.
- **Dispatch mapping** (mirrors `06_litellm_autorouter_v2.ipynb`'s tiers, so the two
  are comparable): `SIMPLE → OLLAMA_MODEL` (local-fast), `MEDIUM → OLLAMA_MODEL_2`
  (local-alt — **the same model tag as the classifier itself**, worth watching for:
  a MEDIUM-tier request gets classified and answered by two separate calls to the
  same underlying model), `COMPLEX`/`REASONING → openai`.
- **This is a semantic judgment, not a reimplementation of Auto Router v2's scoring
  formula.** The rubric in the classifier prompt is a plain natural-language
  description of each tier — it doesn't try to reproduce the heuristic scorer's exact
  token-count/signal-weight thresholds. Disagreement with the exact-boundary eval
  rows (the ones with score-threshold comments) is a real finding about this
  mechanism, not a bug to fix.
- middleware inspects the last user message, classifies it, and
  `handler(request.override(model=...))`
- `create_agent(..., tools=[], middleware=[route])` so one turn = one routed call
- A bad/unparseable classification raises (Pydantic's `Literal` validation on the
  structured output enforces this for free) rather than silently defaulting —
  matches 04/06's "don't fake a route" rule.

Eval below times **only** the decision function (no generation) — now measuring a
real local-model inference call, i.e. this classifier's own "router tax," directly
comparable to Jev's (hosted), RouteLLM's (local `mf` model), and Auto Router v2's
(near-zero heuristic) tax numbers.

Kernel: Python 3.10+.


In [1]:
%pip install langchain langchain-openai langchain-ollama python-dotenv -q


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

from dotenv import load_dotenv

cwd = Path.cwd()
poc_dir = None
root = None
for p in [cwd, *cwd.parents]:
    if (p / "eval_queries.py").exists():
        poc_dir = p
        break
    if (p / "poc" / "eval_queries.py").exists():
        poc_dir = p / "poc"
        break
if poc_dir is None:
    raise FileNotFoundError("eval_queries.py not found — run from route-chatbot/ or route-chatbot/poc/")
sys.path.insert(0, str(poc_dir))

for p in [cwd, *cwd.parents]:
    if (p / ".env").exists() and (p / "main.py").exists():
        root = p
        load_dotenv(p / ".env")
        break
else:
    load_dotenv()

from eval_queries_tiers import EVAL_QUERIES

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3")
OLLAMA_MODEL_2 = os.getenv("OLLAMA_MODEL_2", "llama3")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-3.5-turbo")
TYPESAFE_API_KEY = os.getenv("TYPESAFE_API_KEY", "")

print("poc_dir", poc_dir)
print("eval queries", len(EVAL_QUERIES))
print("ollama", OLLAMA_BASE_URL, OLLAMA_MODEL, "| alt", OLLAMA_MODEL_2)
print("openai model", OPENAI_MODEL, "| key set", bool(OPENAI_API_KEY))
print("typesafe key set", bool(TYPESAFE_API_KEY.strip()))
print("typesafe key set", bool(TYPESAFE_API_KEY.strip()))


poc_dir /Users/tushar/PravarAI/route-chatbot/poc
eval queries 35
ollama http://localhost:11434 llama3.1:8b | alt qwen3.5:9b
openai model gpt-3.5-turbo | key set True
typesafe key set False
typesafe key set False


## Classifier used by middleware


In [3]:
from typing import Literal

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field

# Same rubric as the semantic labels in eval_queries_tiers.py. This is a
# judgment of the task, not Auto Router v2's token/signal score thresholds.
CLASSIFIER_PROMPT = """Classify the user's message into exactly one tier.

SIMPLE: casual conversation, greetings, small talk, or a simple factual question a small local model handles fine.
MEDIUM: some technical vocabulary or a moderately involved question, but no real code and no multi-step reasoning.
COMPLEX: a real coding task (write, fix, or debug code) or nontrivial technical analysis or comparison, especially combined with reasoning.
REASONING: the user explicitly asks for step-by-step logical or mathematical reasoning, a proof, or chained deduction.

Judge the task, not individual words. A word like "code" in a casual question does not make the task COMPLEX."""


class RouteTier(BaseModel):
    """One routing decision. An invalid tier fails validation instead of defaulting."""

    tier: Literal["SIMPLE", "MEDIUM", "COMPLEX", "REASONING"]
    reason: str = Field(description="One sentence explaining the tier.")


ollama_chat = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL)
ollama_alt = ChatOllama(model=OLLAMA_MODEL_2, base_url=OLLAMA_BASE_URL)
openai_chat = ChatOpenAI(model=OPENAI_MODEL, api_key=OPENAI_API_KEY or None)

classifier_kwargs = {
    "model": OLLAMA_MODEL_2,
    "base_url": OLLAMA_BASE_URL,
    "temperature": 0,
}
# qwen3.5 thinks by default. Thinking text is not the JSON schema, so the
# classifier call turns it off. The answer models above are left unchanged.
if "reasoning" in getattr(ChatOllama, "model_fields", {}):
    classifier_kwargs["reasoning"] = False
classifier = ChatOllama(**classifier_kwargs).with_structured_output(
    RouteTier, method="json_schema"
)

# SIMPLE -> local-fast, MEDIUM -> local-alt (same tag as the classifier,
# separate call), COMPLEX and REASONING -> openai.
MODEL_FOR_TIER = {
    "SIMPLE": (ollama_chat, OLLAMA_MODEL),
    "MEDIUM": (ollama_alt, OLLAMA_MODEL_2),
    "COMPLEX": (openai_chat, OPENAI_MODEL),
    "REASONING": (openai_chat, OPENAI_MODEL),
}


def last_user_text(messages) -> str:
    for msg in reversed(list(messages)):
        content = msg.content if hasattr(msg, "content") else msg.get("content", "")
        role = getattr(msg, "type", None) or msg.get("role", "")
        if role in ("human", "user") or getattr(msg, "type", None) == "human":
            return content if isinstance(content, str) else str(content)
    if not messages:
        return ""
    last = messages[-1]
    return last.content if hasattr(last, "content") else str(last)


def decide_route(message: str) -> tuple[str, dict]:
    result = classifier.invoke([
        {"role": "system", "content": CLASSIFIER_PROMPT},
        {"role": "user", "content": message},
    ])
    return result.tier, {"reason": result.reason, "classifier": OLLAMA_MODEL_2}


last_route = {"label": None, "model": None}


@wrap_model_call
def route_models(request, handler):
    text = last_user_text(request.messages)
    tier, extra = decide_route(text)
    chosen, model_name = MODEL_FOR_TIER[tier]
    last_route["label"] = tier
    last_route["model"] = model_name
    last_route["extra"] = extra
    return handler(request.override(model=chosen))


agent = create_agent(
    model=ollama_chat,
    tools=[],
    middleware=[route_models],
)
print(
    "classifier", OLLAMA_MODEL_2,
    "| SIMPLE", OLLAMA_MODEL,
    "| MEDIUM", OLLAMA_MODEL_2,
    "| COMPLEX/REASONING", OPENAI_MODEL,
)


classifier qwen3.5:9b | SIMPLE llama3.1:8b | MEDIUM qwen3.5:9b | COMPLEX/REASONING gpt-3.5-turbo


## Eval (decision function only — no `agent.invoke`)


In [4]:
rows = []
for item in EVAL_QUERIES:
    t0 = time.perf_counter()
    err = None
    predicted = None
    extra = None
    try:
        result = decide_route(item["message"])
        if isinstance(result, tuple):
            predicted = result[0]
            extra = result[1] if len(result) > 1 else None
        else:
            predicted = result
    except Exception as e:
        err = f"{type(e).__name__}: {e}"
    ms = (time.perf_counter() - t0) * 1000
    rows.append({
        "message": item["message"],
        "expected": item["expected"],
        "predicted": predicted,
        "match": predicted == item["expected"],
        "latency_ms": round(ms, 1),
        "error": err,
        "extra": extra,
    })

n = len(rows)
ok = sum(1 for r in rows if r["match"])
errs = sum(1 for r in rows if r["error"])
mean_ms = sum(r["latency_ms"] for r in rows) / n if n else 0
print(f"accuracy {ok}/{n} ({100 * ok / n:.0f}%)  mean latency {mean_ms:.1f} ms  errors {errs}")
print()
for r in rows:
    flag = "OK  " if r["match"] else "MISS"
    extra = f"  {r['extra']}" if r["extra"] else ""
    err = f"  ERR {r['error']}" if r["error"] else ""
    print(f"  [{flag}] {r['latency_ms']:7.1f} ms  exp={r['expected']:7} pred={r['predicted']}  {r['message'][:70]}{extra}{err}")


accuracy 24/35 (69%)  mean latency 3220.2 ms  errors 0

  [OK  ]  2665.2 ms  exp=SIMPLE  pred=SIMPLE  hi there  {'reason': "The message 'hi there' is a casual greeting, which falls under small talk handled by a simple local model.", 'classifier': 'qwen3.5:9b'}
  [OK  ]  2456.0 ms  exp=SIMPLE  pred=SIMPLE  hello  {'reason': "The message 'hello' is a greeting and falls under casual conversation, which is handled by the SIMPLE tier.", 'classifier': 'qwen3.5:9b'}
  [OK  ]  2506.4 ms  exp=SIMPLE  pred=SIMPLE  hey  {'reason': "The message 'hey' is a casual greeting and falls under small talk, which is handled by a simple local model.", 'classifier': 'qwen3.5:9b'}
  [OK  ]  2118.7 ms  exp=SIMPLE  pred=SIMPLE  good morning  {'reason': "The message 'good morning' is a standard greeting and falls under casual conversation.", 'classifier': 'qwen3.5:9b'}
  [OK  ]  2357.5 ms  exp=SIMPLE  pred=SIMPLE  thanks  {'reason': "The message 'thanks' is a casual expression of gratitude, fitting the definitio

## Notes (fill during the experiment)

- Middleware is the integration point: routing stays out of the agent graph.
- `decide_route` calls `qwen` (`OLLAMA_MODEL_2`) and returns a tier. The eval times only that call, so latency is the classifier's router tax. `route_models` maps the tier onto a chat model and does not re-classify.
- Labels come from `eval_queries_tiers.py`. The boundary rows are scored against Auto Router v2's heuristic thresholds; disagreement with those rows is a finding about semantic judgment versus that formula.
- `agent.invoke` latency includes generation on top of the classifier call.


In [5]:
GENERATE = True

if GENERATE:
    for item in EVAL_QUERIES[:]:
        result = agent.invoke({"messages": [{"role": "user", "content": item["message"]}]})
        print("---", item["message"], "->", last_route["label"], last_route.get("model"))
        msgs = result.get("messages", [])
        last = msgs[-1] if msgs else result
        content = getattr(last, "content", last)
        print(str(content)[:400])
        print()
else:
    print("GENERATE is False — routing only. Flip it to call agent.invoke.")


--- hi there -> SIMPLE llama3.1:8b
How's it going? Is there something I can help you with or would you like to chat?

--- hello -> SIMPLE llama3.1:8b
Hello! How are you today? Is there something I can help you with or would you like to chat?

--- hey -> SIMPLE llama3.1:8b
How's it going? Is there something I can help you with or would you like to chat?

--- good morning -> SIMPLE llama3.1:8b
Good morning! Hope you're having a great start to the day! How can I help you today?

--- thanks -> SIMPLE llama3.1:8b
You're welcome! Is there anything else I can help you with?

--- how are you -> SIMPLE llama3.1:8b
I'm just a computer program, so I don't have feelings or emotions like humans do. I'm functioning properly and ready to assist you with any questions or tasks you may have! How about you? How's your day going so far?

--- compare merge sort and quick sort -> MEDIUM qwen3.5:9b
Here is a detailed comparison between **Merge Sort** and **Quick Sort**, two of the most popular efficient sor